# PaCMAP visualization of ModernMolBERT embeddings

In [7]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pacmap
import pandas as pd

## Config

In [ ]:
EMBED_DIR = Path("../../outputs/visualize/best_standard")

# Source parquet used when embedding — needed to join extra property columns.
SOURCE_PARQUET = Path("../../data/pretrain/chembl36_selfies/valid.parquet")

# Column from metadata.parquet to color by.
COLOR_COLUMN = "alogp"

# Additional property columns to join from SOURCE_PARQUET.
# Any subset of the list below — they are merged by chembl_id at load time.
EXTRA_COLUMNS = [
    "psa",
    "hba",
    "hbd",
    "qed_weighted",
    "mw_freebase",
    "rtb",
    "aromatic_rings",
    "heavy_atoms",
    "num_ro5_violations",
]

# Display labels for the color legend.
COLUMN_LABELS = {
    "alogp": "ALogP (lipophilicity)",
    "psa": "Polar surface area (Å²)",
    "hba": "H-bond acceptors",
    "hbd": "H-bond donors",
    "qed_weighted": "QED (drug-likeness)",
    "mw_freebase": "Molecular weight (Da)",
    "rtb": "Rotatable bonds",
    "aromatic_rings": "Aromatic rings",
    "heavy_atoms": "Heavy atom count",
    "num_ro5_violations": "Ro5 violations",
    "is_valid": "Valid SELFIES",
}

## Load

In [11]:
embeddings = np.load(EMBED_DIR / "embeddings.npy")
meta = pd.read_parquet(EMBED_DIR / "metadata.parquet")

print(f"Embeddings: {embeddings.shape}")
print(f"Metadata columns: {meta.columns.tolist()}")

Embeddings: (10000, 512)
Metadata columns: ['chembl_id', 'smiles_canonical_clean', 'selfies', 'alogp', 'canonical_smiles', 'standard_inchi_key', 'is_valid', 'embedding_row']


In [ ]:
if EXTRA_COLUMNS and SOURCE_PARQUET.exists():
    available = set(pd.read_parquet(SOURCE_PARQUET, columns=["chembl_id"]).columns)
    cols_to_load = [c for c in EXTRA_COLUMNS if c not in meta.columns]
    if cols_to_load:
        props = pd.read_parquet(SOURCE_PARQUET, columns=["chembl_id"] + cols_to_load)
        meta = meta.merge(props, on="chembl_id", how="left")
        print(f"Joined {cols_to_load} from source parquet.")
elif EXTRA_COLUMNS:
    print(f"Warning: SOURCE_PARQUET not found at {SOURCE_PARQUET}")

## Run PaCMAP

In [ ]:
reducer = pacmap.PaCMAP(
    n_components=2,
    n_neighbors=5,
    MN_ratio=0.0,
    FP_ratio=12.0,
    verbose=True,
)

coords = reducer.fit_transform(embeddings, init="pca")
print(f"PaCMAP output: {coords.shape}")

## Visualize

In [ ]:
if COLOR_COLUMN not in meta.columns:
    raise ValueError(f"{COLOR_COLUMN!r} not in metadata. Available: {meta.columns.tolist()}")

color_series = meta[COLOR_COLUMN]
is_numeric = pd.api.types.is_numeric_dtype(color_series)
col_label = COLUMN_LABELS.get(COLOR_COLUMN, COLOR_COLUMN)

fig, ax = plt.subplots(figsize=(7, 6))

if is_numeric:
    hb = ax.hexbin(
        coords[:, 0],
        coords[:, 1],
        C=color_series.values,
        reduce_C_function=np.mean,
        gridsize=80,
        cmap="RdBu_r",
        linewidths=0.15,
        mincnt=1,
    )
    cb = fig.colorbar(hb, ax=ax, shrink=0.6, aspect=20, pad=0.02)
    cb.set_label(col_label, fontsize=10)
    cb.ax.tick_params(labelsize=8)
else:
    categories = color_series.astype(str)
    unique_cats = sorted(categories.unique())
    palette = plt.cm.get_cmap("tab20", len(unique_cats))
    cat_to_int = {c: i for i, c in enumerate(unique_cats)}
    c_vals = np.array([cat_to_int[c] for c in categories])
    hb = ax.hexbin(
        coords[:, 0],
        coords[:, 1],
        C=c_vals,
        reduce_C_function=lambda v: float(np.bincount(v.astype(int)).argmax()),
        gridsize=80,
        cmap=palette,
        linewidths=0.15,
        mincnt=1,
    )
    handles = [
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor=palette(i / max(len(unique_cats) - 1, 1)),
            markersize=7,
            label=c,
        )
        for i, c in enumerate(unique_cats)
    ]
    ax.legend(
        handles=handles,
        title=col_label,
        bbox_to_anchor=(1.01, 1),
        loc="upper left",
        fontsize=8,
        title_fontsize=9,
        frameon=False,
    )

ax.set_title(f"PaCMAP — {col_label}", fontsize=13, pad=10)
ax.set_xlabel("PaCMAP 1", fontsize=11)
ax.set_ylabel("PaCMAP 2", fontsize=11)
ax.set_xticks([])
ax.set_yticks([])
ax.spines[["top", "right", "left", "bottom"]].set_visible(False)

plt.tight_layout()
plt.show()